### Two-Stage Hourly Classification Model (3 classes)

**Stage 1:** Binary classifier (zero vs non-zero)  
**Stage 2:** Binary classifier (low vs high) on non-zero subset only  

3 classes: **zero** (0 trips), **low** (1-3 trips), **high** (4+ trips)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Load features saved by notebook 08
df = pd.read_csv('../Project_datasets/hourly_features.csv')
df['time_bin'] = pd.to_datetime(df['time_bin'])
df = df.sort_values('time_bin').reset_index(drop=True)

# Drop first week of each season (same logic as notebook 08 cell 18)
season_starts = []
for year in range(2014, 2018):
    start = pd.Timestamp(f'{year}-04-01')
    end = start + pd.Timedelta(hours=56 * 3 - 1)
    season_starts.append((start, end))

mask_season_start = pd.Series(False, index=df.index)
for start, end in season_starts:
    mask_season_start |= (df['time_bin'] >= start) & (df['time_bin'] <= end)

df_model = df[~mask_season_start].copy()
print(f"Rows after dropping season starts: {len(df_model)}")

# Feature columns (same as notebook 08)
exclude_cols = ['time_bin', 'trip_count', 'avg_duration', 'member_count']
feature_cols = [c for c in df_model.columns if c not in exclude_cols]
print(f"Features: {len(feature_cols)}")

# Time-based train/test split
df_model['year'] = df_model['time_bin'].dt.year
train_full = df_model[df_model['year'] < 2017].dropna(subset=feature_cols)
test_full = df_model[df_model['year'] == 2017].dropna(subset=feature_cols)
print(f"Train: {len(train_full)}, Test: {len(test_full)}")

In [ ]:
# ── Stage 1: Binary classifier (zero vs non-zero) ──

y_train_s1 = (train_full['trip_count'] > 0).astype(int)
y_test_s1 = (test_full['trip_count'] > 0).astype(int)
X_train_s1 = train_full[feature_cols]
X_test_s1 = test_full[feature_cols]

smote_s1 = SMOTE(random_state=42)
X_train_s1_bal, y_train_s1_bal = smote_s1.fit_resample(X_train_s1, y_train_s1)
print(f"Stage 1 SMOTE: {len(X_train_s1_bal)} rows")
print(f"Class balance: {pd.Series(y_train_s1_bal).value_counts().to_dict()}")

model_s1 = GradientBoostingClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42
)
model_s1.fit(X_train_s1_bal, y_train_s1_bal)

y_pred_s1 = model_s1.predict(X_test_s1)
acc_s1 = accuracy_score(y_test_s1, y_pred_s1)
print(f"\nStage 1 Accuracy: {acc_s1:.4f}")
print(classification_report(y_test_s1, y_pred_s1, target_names=['zero', 'non-zero']))
print("Confusion matrix:")
print(confusion_matrix(y_test_s1, y_pred_s1))

In [ ]:
# ── Stage 2: Binary classifier (low vs high) on non-zero rows only ──

# Filter to non-zero rows
train_nz = train_full[train_full['trip_count'] > 0].copy()
test_nz = test_full[test_full['trip_count'] > 0].copy()

# Create 2-class target: low (1-3 trips), high (4+ trips)
train_nz['nz_bin'] = pd.cut(
    train_nz['trip_count'], bins=[0, 3, np.inf], labels=['low', 'high']
)
test_nz['nz_bin'] = pd.cut(
    test_nz['trip_count'], bins=[0, 3, np.inf], labels=['low', 'high']
)

le_s2 = LabelEncoder()
le_s2.fit(['high', 'low'])
y_train_s2 = le_s2.transform(train_nz['nz_bin'])
y_test_s2 = le_s2.transform(test_nz['nz_bin'])
X_train_s2 = train_nz[feature_cols]
X_test_s2 = test_nz[feature_cols]

print(f"Non-zero train: {len(train_nz)}, test: {len(test_nz)}")
print(f"Class distribution (train):")
print(train_nz['nz_bin'].value_counts().sort_index())

smote_s2 = SMOTE(random_state=42)
X_train_s2_bal, y_train_s2_bal = smote_s2.fit_resample(X_train_s2, y_train_s2)

model_s2 = GradientBoostingClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42
)
model_s2.fit(X_train_s2_bal, y_train_s2_bal)

y_pred_s2 = model_s2.predict(X_test_s2)
acc_s2 = accuracy_score(y_test_s2, y_pred_s2)
print(f"\nStage 2 Accuracy: {acc_s2:.4f}")
print(classification_report(y_test_s2, y_pred_s2, target_names=le_s2.classes_))
print("Confusion matrix:")
print(confusion_matrix(y_test_s2, y_pred_s2))

In [ ]:
# ── Combined evaluation: assemble 3-class predictions ──

# Stage 1 on full test set
s1_pred = model_s1.predict(test_full[feature_cols])

# Initialize combined predictions
combined_pred = pd.Series('zero', index=test_full.index)

# For rows predicted non-zero by Stage 1, run Stage 2
nonzero_mask = s1_pred == 1
if nonzero_mask.sum() > 0:
    s2_pred = model_s2.predict(test_full.loc[nonzero_mask, feature_cols])
    combined_pred.loc[nonzero_mask] = le_s2.inverse_transform(s2_pred)

# Build true 3-class labels
true_3class = pd.cut(
    test_full['trip_count'],
    bins=[-1, 0, 3, np.inf],
    labels=['zero', 'low', 'high']
)

le_3 = LabelEncoder()
le_3.fit(['high', 'low', 'zero'])
y_true_enc = le_3.transform(true_3class)
y_pred_enc = le_3.transform(combined_pred)

acc_combined = accuracy_score(y_true_enc, y_pred_enc)
f1_combined = f1_score(y_true_enc, y_pred_enc, average='weighted')

print("=" * 55)
print("TWO-STAGE COMBINED RESULTS (3-class)")
print("=" * 55)
print(f"Overall Accuracy:  {acc_combined:.4f}")
print(f"Weighted F1:       {f1_combined:.4f}")
print()
print(classification_report(y_true_enc, y_pred_enc, target_names=le_3.classes_))

# Confusion matrix heatmap
cm = confusion_matrix(y_true_enc, y_pred_enc)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le_3.classes_, yticklabels=le_3.classes_, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title(f'Two-Stage Combined (3-class) — Accuracy = {acc_combined:.4f}')
plt.tight_layout()
plt.show()

In [ ]:
# ── Cross-validation (TimeSeriesSplit, 5 folds) ──

df_cv = df_model.dropna(subset=feature_cols)
X_all = df_cv[feature_cols]
trip_counts_all = df_cv['trip_count']

tscv = TimeSeriesSplit(n_splits=5)
cv_scores = []

for fold, (train_idx, val_idx) in enumerate(tscv.split(X_all)):
    X_tr, X_val = X_all.iloc[train_idx], X_all.iloc[val_idx]
    tc_tr, tc_val = trip_counts_all.iloc[train_idx], trip_counts_all.iloc[val_idx]

    # Stage 1: zero vs non-zero
    y_tr_s1 = (tc_tr > 0).astype(int)
    smote_cv1 = SMOTE(random_state=42)
    X_tr_s1_bal, y_tr_s1_bal = smote_cv1.fit_resample(X_tr, y_tr_s1)
    cv_m1 = GradientBoostingClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42)
    cv_m1.fit(X_tr_s1_bal, y_tr_s1_bal)

    # Stage 2: low/high on non-zero training rows
    nz_mask_tr = tc_tr > 0
    nz_labels_tr = pd.cut(tc_tr[nz_mask_tr], bins=[0, 3, np.inf], labels=['low', 'high'])
    le_cv = LabelEncoder()
    le_cv.fit(['high', 'low'])
    y_tr_s2 = le_cv.transform(nz_labels_tr)
    smote_cv2 = SMOTE(random_state=42)
    X_tr_s2_bal, y_tr_s2_bal = smote_cv2.fit_resample(X_tr[nz_mask_tr], y_tr_s2)
    cv_m2 = GradientBoostingClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42)
    cv_m2.fit(X_tr_s2_bal, y_tr_s2_bal)

    # Combined prediction on validation set
    s1_p = cv_m1.predict(X_val)
    combined = pd.Series('zero', index=X_val.index)
    nz_pred_mask = s1_p == 1
    if nz_pred_mask.sum() > 0:
        s2_p = cv_m2.predict(X_val[nz_pred_mask])
        combined.loc[combined.index[nz_pred_mask]] = le_cv.inverse_transform(s2_p)

    # True 3-class labels
    true_labels = pd.cut(tc_val, bins=[-1, 0, 3, np.inf], labels=['zero', 'low', 'high'])
    le_3cv = LabelEncoder()
    le_3cv.fit(['high', 'low', 'zero'])
    y_true_cv = le_3cv.transform(true_labels)
    y_pred_cv = le_3cv.transform(combined)

    fold_acc = accuracy_score(y_true_cv, y_pred_cv)
    fold_f1 = f1_score(y_true_cv, y_pred_cv, average='weighted')
    cv_scores.append({'fold': fold + 1, 'Accuracy': fold_acc, 'F1_weighted': fold_f1})
    print(f"Fold {fold+1}: Accuracy = {fold_acc:.4f}, F1 (weighted) = {fold_f1:.4f}")

cv_df = pd.DataFrame(cv_scores)
print(f"\nMean CV Accuracy: {cv_df['Accuracy'].mean():.4f} +/- {cv_df['Accuracy'].std():.4f}")
print(f"Mean CV F1 (weighted): {cv_df['F1_weighted'].mean():.4f} +/- {cv_df['F1_weighted'].std():.4f}")

In [ ]:
# ── Feature importance (top 20 for each stage) ──

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

for ax, model, title in [
    (axes[0], model_s1, 'Stage 1: Zero vs Non-zero'),
    (axes[1], model_s2, 'Stage 2: Low / Mid / High'),
]:
    imp = pd.DataFrame({
        'feature': feature_cols,
        'importance': model.feature_importances_
    }).sort_values('importance', ascending=False)

    imp.head(20).plot.barh(x='feature', y='importance', ax=ax, legend=False)
    ax.set_xlabel('Importance')
    ax.set_title(f'Top 20 Features — {title}')
    ax.invert_yaxis()

plt.tight_layout()
plt.show()

# Print top 10 for each stage
for model, title in [(model_s1, 'Stage 1'), (model_s2, 'Stage 2')]:
    imp = pd.DataFrame({'feature': feature_cols, 'importance': model.feature_importances_})
    imp = imp.sort_values('importance', ascending=False)
    print(f"\nTop 10 features ({title}):")
    print(imp.head(10).to_string(index=False))

In [ ]:
# ── Save results ──

results_df = pd.DataFrame([{
    'model': 'TwoStage_GradientBoosting',
    'stage1_accuracy': round(acc_s1, 4),
    'stage2_accuracy': round(acc_s2, 4),
    'combined_accuracy': round(acc_combined, 4),
    'combined_F1_weighted': round(f1_combined, 4),
    'cv_mean_accuracy': round(cv_df['Accuracy'].mean(), 4),
    'cv_mean_F1_weighted': round(cv_df['F1_weighted'].mean(), 4),
}])

results_df.to_csv('../Project_datasets/model_results_hourly_two_stage.csv', index=False)
print("Saved model_results_hourly_two_stage.csv")
print()
print(results_df.to_string(index=False))